# Missing patch vs origin review

Use this notebook to manually compare an unassociated patch against candidate origin images. It does not modify the accepted metadata or fold files.

Workflow:
1. Run all cells.
2. Type a patch id such as `p0010`, then press **Enter** in the patch box or click **Compare**.
3. Inspect the patch, best origin crop, difference image, and origin overlay.
4. Fill the decision fields and click **Save decision**.

Decisions are written to `results/phase0_missing_patch_review/notebook_manual_decisions.csv` as candidate-only review notes.

In [4]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("ipywidgets is required for this review notebook. Install/enable it in this environment.") from exc

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "scripts").exists():
    raise RuntimeError(f"Could not find repo root from {Path.cwd()}")

sys.path.insert(0, str(REPO_ROOT / "scripts"))

from src.phase0.patch_origin_registration import (
    candidate_origin_ids,
    crop_origin_region,
    make_difference_image,
    make_overlay,
    match_patch_to_origin,
    read_rgb,
)

QUEUE_PATH = REPO_ROOT / "results/phase0_missing_patch_review/similar_patch_groups/missing_patch_review_queue_with_similarity_groups.csv"
if not QUEUE_PATH.exists():
    QUEUE_PATH = REPO_ROOT / "results/phase0_missing_patch_review/missing_patch_review_queue.csv"

ORIGIN_IMAGE_DIR = REPO_ROOT / "data/ndb_ufes/origin_level/images"
DECISIONS_PATH = REPO_ROOT / "results/phase0_missing_patch_review/notebook_manual_decisions.csv"
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

queue = pd.read_csv(QUEUE_PATH).sort_values("patch_number").reset_index(drop=True)
queue["patch"] = queue["patch"].astype(str)

def resolve_path(path_value):
    path = Path(str(path_value))
    return path if path.is_absolute() else REPO_ROOT / path

print(f"Loaded {len(queue)} missing patches from {QUEUE_PATH.relative_to(REPO_ROOT)}")
print(f"Manual decisions will be saved to {DECISIONS_PATH.relative_to(REPO_ROOT)}")

Loaded 677 missing patches from results/phase0_missing_patch_review/similar_patch_groups/missing_patch_review_queue_with_similarity_groups.csv
Manual decisions will be saved to results/phase0_missing_patch_review/notebook_manual_decisions.csv


In [5]:
def row_for_patch(patch_id: str) -> pd.Series:
    patch_id = str(patch_id).strip()
    match = queue[queue["patch"] == patch_id]
    if match.empty:
        raise ValueError(f"Patch not found in review queue: {patch_id}")
    return match.iloc[0]


def all_numeric_origin_ids() -> list[str]:
    return sorted(path.stem for path in ORIGIN_IMAGE_DIR.glob("[0-9][0-9][0-9][0-9].png"))


def patchless_origin_ids() -> list[str]:
    path = REPO_ROOT / "results/phase0_recovery_validation/patchless_origin_candidates.csv"
    if not path.exists():
        return []
    table = pd.read_csv(path)
    if "origin_id" not in table.columns:
        return []
    return sorted(str(value).zfill(4) for value in table["origin_id"].dropna().tolist())


def candidate_ids_for_review(row: pd.Series, extra_origin_text: str = "", search_scope: str = "context") -> list[str]:
    if search_scope == "all origins":
        ids = all_numeric_origin_ids()
    elif search_scope == "patchless origins":
        ids = patchless_origin_ids()
    else:
        ids = candidate_origin_ids(row, ORIGIN_IMAGE_DIR)
    for part in str(extra_origin_text).replace(",", " ").split():
        origin_id = part.strip().zfill(4)
        if origin_id.isdigit() and origin_id not in ids:
            ids.append(origin_id)
    return ids


def show_patch_only(patch_id: str):
    row = row_for_patch(patch_id)
    patch_path = resolve_path(row["raw_patch_path"])
    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(read_rgb(patch_path))
    ax.set_title(f"{patch_id} while origin search runs")
    ax.axis("off")
    plt.show()


def compare_patch_to_candidates(patch_id: str, extra_origin_text: str = "", search_scope: str = "context", progress=None, scales=(1.0,)) -> pd.DataFrame:
    row = row_for_patch(patch_id)
    patch_path = resolve_path(row["raw_patch_path"])
    rows = []
    origin_ids = candidate_ids_for_review(row, extra_origin_text, search_scope=search_scope)
    if progress is not None:
        progress.max = len(origin_ids)
        progress.value = 0
        progress.description = "Searching"
        progress.bar_style = "info"
    for index, origin_id in enumerate(origin_ids, start=1):
        if progress is not None:
            progress.value = index - 1
            progress.description = f"{index}/{len(origin_ids)}"
        origin_path = ORIGIN_IMAGE_DIR / f"{origin_id}.png"
        if not origin_path.exists():
            continue
        match = match_patch_to_origin(patch_path, origin_path, scales=scales)
        rows.append({
            "patch": row["patch"],
            "patch_number": int(row["patch_number"]),
            "patch_path": str(patch_path),
            "candidate_origin_id": origin_id,
            "origin_path": str(origin_path),
            "folder_label": row.get("folder_label", ""),
            "similarity_group_id": row.get("similarity_group_id", ""),
            "similarity_group_size": row.get("similarity_group_size", ""),
            **match,
        })
    if progress is not None:
        progress.value = len(origin_ids)
        progress.description = "Done"
        progress.bar_style = "success"
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("match_quality_score", ascending=False).reset_index(drop=True)


def show_comparison(results: pd.DataFrame, max_candidates: int = 3):
    if results.empty:
        print("No candidate origin images found for this patch.")
        return
    rows = results.head(max_candidates).to_dict("records")
    fig, axes = plt.subplots(len(rows), 4, figsize=(15, 4 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for axis_row, row in zip(axes, rows):
        patch_image = read_rgb(Path(row["patch_path"]))
        crop_image = crop_origin_region(Path(row["origin_path"]), row)
        diff_image = make_difference_image(Path(row["patch_path"]), Path(row["origin_path"]), row)
        overlay_image = make_overlay(Path(row["origin_path"]), row, size=(420, 320))
        title = (
            f"origin {row['candidate_origin_id']} | {row.get('match_verdict', '')} | "
            f"q {row.get('match_quality_score', 0):.3f} | ncc {row['ncc']:.3f} | "
            f"lab {row.get('lab_mean_delta', 0):.1f} | tissue_delta {row.get('tissue_fraction_delta', 0):.2f} | "
            f"crop {row.get('origin_width', 0)}x{row.get('origin_height', 0)} scale {row.get('scale', 1):.2f}"
        )
        for ax, image, label in zip(axis_row, [patch_image, crop_image, diff_image, overlay_image], ["patch", "best crop", "abs diff", title]):
            ax.imshow(image)
            ax.set_title(label)
            ax.axis("off")
    plt.tight_layout()
    display(results[["candidate_origin_id", "match_verdict", "match_quality_score", "template_score", "ncc", "lab_mean_delta", "tissue_fraction_delta", "mae", "origin_x", "origin_y", "origin_width", "origin_height", "scale"]].head(max_candidates))
    plt.show()


def save_manual_decision(record: dict) -> None:
    new_row = pd.DataFrame([record])
    if DECISIONS_PATH.exists():
        old = pd.read_csv(DECISIONS_PATH)
        table = pd.concat([old, new_row], ignore_index=True)
    else:
        table = new_row
    table.to_csv(DECISIONS_PATH, index=False)

In [6]:
patch_box = widgets.Text(value=str(queue.iloc[0]["patch"]), description="Patch", continuous_update=False)
extra_origins_box = widgets.Text(value="", description="Origins", placeholder="optional: 0010 0041")
search_scope_dropdown = widgets.Dropdown(
    options=["context", "patchless origins", "all origins"],
    value="context",
    description="Search",
)
allow_scaled_search_box = widgets.Checkbox(value=False, description="allow scaled search")
max_candidates_slider = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Show")
compare_button = widgets.Button(description="Compare", button_style="primary")
previous_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
progress_bar = widgets.IntProgress(value=0, min=0, max=1, description="Idle", bar_style="")

decision_dropdown = widgets.Dropdown(
    options=["unsure", "accept_origin", "reject_candidates", "needs_more_candidates", "skip"],
    value="unsure",
    description="Decision",
)
verified_origin_box = widgets.Text(value="", description="Verified")
verified_label_box = widgets.Text(value="", description="Label")
reviewer_box = widgets.Text(value="", description="Reviewer")
notes_box = widgets.Textarea(value="", description="Notes", layout=widgets.Layout(width="650px", height="80px"))
save_button = widgets.Button(description="Save decision", button_style="success")
output = widgets.Output()

current_results = {"table": pd.DataFrame()}


def current_index():
    matches = queue.index[queue["patch"] == patch_box.value.strip()].tolist()
    return matches[0] if matches else 0


def load_patch_at(index: int):
    index = max(0, min(index, len(queue) - 1))
    patch_box.value = str(queue.iloc[index]["patch"])
    run_compare()


def run_compare(_=None):
    with output:
        clear_output(wait=True)
        try:
            row = row_for_patch(patch_box.value)
            print(
                f"{row['patch']} | row {current_index() + 1}/{len(queue)} | "
                f"folder label: {row.get('folder_label', '')} | "
                f"similarity group: {row.get('similarity_group_id', '') or 'none'} "
                f"n={row.get('similarity_group_size', '')}"
            )
            print(
                f"candidate context: prev origin {row.get('previous_accepted_origin_id', '')}, "
                f"next origin {row.get('next_accepted_origin_id', '')}, "
                f"proposed origin {row.get('proposed_correct_origin_id', '')}"
            )
            print(f"search scope: {search_scope_dropdown.value}")
            if search_scope_dropdown.value == "all origins":
                print("Searching all origin images can take 1-2 minutes. The patch is shown first so the cell does not look empty.")
                show_patch_only(patch_box.value)
            scales = (1.0, 0.75, 0.5, 0.25) if allow_scaled_search_box.value else (1.0,)
            print(f"scale mode: {'scaled search allowed' if allow_scaled_search_box.value else 'exact 512x512 crop only'}")
            results = compare_patch_to_candidates(
                patch_box.value,
                extra_origins_box.value,
                search_scope=search_scope_dropdown.value,
                progress=progress_bar,
                scales=scales,
            )
            current_results["table"] = results
            if not results.empty:
                best = results.iloc[0]
                print(
                    f"best verdict: {best.get('match_verdict', '')} | "
                    f"quality {best.get('match_quality_score', 0):.3f} | "
                    f"grayscale ncc {best.get('ncc', 0):.3f} | "
                    f"LAB mean delta {best.get('lab_mean_delta', 0):.1f} | "
                    f"tissue delta {best.get('tissue_fraction_delta', 0):.2f}"
                )
                if best.get("match_verdict", "") == "weak_reject":
                    print("Interpretation: best candidate is still visually weak; do not accept without manual evidence.")
            show_comparison(results, max_candidates=max_candidates_slider.value)
        except Exception as exc:
            current_results["table"] = pd.DataFrame()
            print(f"Could not compare patch: {exc}")


def save_decision(_=None):
    with output:
        row = row_for_patch(patch_box.value)
        best = current_results["table"].iloc[0].to_dict() if not current_results["table"].empty else {}
        record = {
            "patch": row["patch"],
            "patch_number": int(row["patch_number"]),
            "decision": decision_dropdown.value,
            "verified_origin_id": verified_origin_box.value.strip().zfill(4) if verified_origin_box.value.strip() else "",
            "verified_diagnosis": verified_label_box.value.strip(),
            "reviewer": reviewer_box.value.strip(),
            "notes": notes_box.value.strip(),
            "best_candidate_origin_id": best.get("candidate_origin_id", ""),
            "best_template_score": best.get("template_score", ""),
            "best_ncc": best.get("ncc", ""),
            "best_mae": best.get("mae", ""),
            "best_match_quality_score": best.get("match_quality_score", ""),
            "best_match_verdict": best.get("match_verdict", ""),
            "best_lab_mean_delta": best.get("lab_mean_delta", ""),
            "best_tissue_fraction_delta": best.get("tissue_fraction_delta", ""),
            "search_scope": search_scope_dropdown.value,
            "allow_scaled_search": allow_scaled_search_box.value,
            "candidate_policy": "candidate_only_manual_verified",
        }
        save_manual_decision(record)
        print(f"Saved decision for {record['patch']} to {DECISIONS_PATH.relative_to(REPO_ROOT)}")


compare_button.on_click(run_compare)
patch_box.observe(lambda change: run_compare() if change.get("name") == "value" else None, names="value")
previous_button.on_click(lambda _: load_patch_at(current_index() - 1))
next_button.on_click(lambda _: load_patch_at(current_index() + 1))
save_button.on_click(save_decision)

controls = widgets.VBox([
    widgets.HBox([patch_box, search_scope_dropdown, extra_origins_box, max_candidates_slider, allow_scaled_search_box, compare_button]),
    progress_bar,
    widgets.HBox([previous_button, next_button]),
    widgets.HBox([decision_dropdown, verified_origin_box, verified_label_box, reviewer_box, save_button]),
    notes_box,
])

display(controls, output)
run_compare()

Output()